# 2026 COMP90042 Project Group 141 Baseline
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

In [25]:
# Install the requirements in Google Colab
!pip install -q -U transformers accelerate datasets evaluate

In [26]:
# Authenticate to Hugging Face
from huggingface_hub import login

login()

In [27]:
# setting device on GPU if available, else CPU
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

Using device: cuda



# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


In [28]:
# load data from Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
from pathlib import Path

# base data path
data_path = Path("/content/drive/MyDrive/Colab Notebooks/COMP90042NLP/COMP90042_Assignment3/data")

# all file paths
data_path_train_claims = data_path / "train-claims.json"
data_path_evidence = data_path / "evidence.json"
data_path_dev_claims = data_path / "dev-claims.json"
data_path_dev_baseline_claims = data_path / "dev-claims-baseline.json"
data_path_test_claims = data_path / "test-claims-unlabelled.json"


In [30]:
import json

with open(data_path_train_claims) as file:
    train_claims = json.load(file)
# train_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_evidence) as file:
    evidence = json.load(file)
# evidence is a dictionary with evidence_id as key and the actual text
# as value.

with open(data_path_dev_claims) as file:
    dev_claims = json.load(file)
# dev_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_dev_baseline_claims) as file:
    dev_baseline_claims = json.load(file)
# dev_baseline_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores claim_text, claim_label and evidences.

with open(data_path_test_claims) as file:
    test_claims = json.load(file)
# test_claims is a dictionary with clailm_id as key and a dictionary as
# value where it stores just claim_text.

In [58]:
import re

def clean_text(text):
    text = text.lower()
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text

def process_data(data, retrieve_train=False, classification_train=False):
    """Process claims into task-specific rows.

    - retrieve_train=True:
      output: list of {claim_id, claim_text}
    - classification_train=True:
      output: list of {claim_id, claim_text, label, evidences, gold_input}
    Exactly one mode should be True.
    """
    processed_data = []
    for claim_id, value in data.items():
        claim_text = clean_text(value["claim_text"])

        if retrieve_train:
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
            })
        elif classification_train:
            evidence_ids = list(value.get("evidences", []))
            evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]
            gold_input = (
                f"Claim: {claim_text} \n\nEvidence:\n"
                + "\n".join([f"- {e}" for e in evidence_texts])
            )
            processed_data.append({
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidences": evidence_ids,
                "label": value.get("claim_label"),
                "gold_input": gold_input,
            })

    return processed_data

def add_gold_input_for_records(records):
    """For list records with evidences, append/build `gold_input` in place."""
    for record in records:
        claim_text = clean_text(record["claim_text"])
        evidence_ids = list(record.get("evidences", []))
        evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids if eid in evidence]

        gold_input = (
            f"Claim: {claim_text} \n\nEvidence:\n"
            + "\n".join([f"- {e}" for e in evidence_texts])
        )

        record["gold_input"] = gold_input

    return records


# Retrieval training/inference rows.
processed_train_retrieve_data = process_data(train_claims, retrieve_train=True)
processed_dev_retrieve_data = process_data(dev_claims, retrieve_train=True)

# Classification training rows built from gold evidences.
processed_train_data = process_data(train_claims, classification_train=True)
processed_dev_data = process_data(dev_claims, classification_train=True)
# processed_train_data, processed_dev_data and processed_dev_baseline_data are lists
# with different claims stored as a dictionary with claim_id, claim_text, label, evidence_ids
# and gold_input. gold_input is a input format for classification.

processed_test_data = process_data(test_claims, retrieve_train=True)
# processed_test_data is a list with different claims stored as a dictionary with
# just claim_id and claim_text.

In [47]:
all_evidence_ids = list(evidence.keys())
all_evidence_texts = [clean_text(evidence[eid]) for eid in all_evidence_ids]
# all_evidence_ids and all_evidence_texts are coresponding lists of all the evidence passages.

In [48]:
print(f"all_evidence_ids.length = {len(all_evidence_ids)}")
print(f"len(processed_train_data) = {len(processed_train_data)}")
print(f"len(processed_dev_data) = {len(processed_dev_data)}")
print(f"len(processed_test_data) = {len(processed_test_data)}")
print(processed_train_data[0])
print(processed_train_data[0].get("gold_input"))

all_evidence_ids.length = 1208827
len(processed_train_data) = 1228
len(processed_dev_data) = 154
len(processed_test_data) = 153
{'claim_id': 'claim-1937', 'claim_text': 'not only is there no scientific evidence that co2 is a pollutant, higher co2 concentrations actually help ecosystems support more plant and animal life.', 'evidences': ['evidence-442946', 'evidence-1194317', 'evidence-12171'], 'label': 'DISPUTED', 'gold_input': 'Claim: not only is there no scientific evidence that co2 is a pollutant, higher co2 concentrations actually help ecosystems support more plant and animal life. \n\nEvidence:\n- at very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal life, so raising the concentration to 10,000 ppm (1%) or higher for several hours will eliminate pests such as whiteflies and spider mites in a greenhouse.\n- plants can grow as much as 50 percent faster in concentrations of 1,000 ppm co 2 when compared with ambient condit

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)


## 2.1 Evidence Retrieval
Given claim_text, output evidence_ids

In [55]:
# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import copy
import numpy as np
import evaluate
from datasets import Dataset
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback
from transformers import DataCollatorWithPadding
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

In [35]:
# Build TF-IDF index over all evidence passages
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
evidence_tfidf = vectorizer.fit_transform(all_evidence_texts)

In [51]:
# Input: claim_text
# Return: top-k evidence_ids
def retrieve_topk_evidence_ids(claim_text, top_k=5):
    # Vectorize the claim with the same fitted TF-IDF vocabulary.
    claim_clean = clean_text(claim_text)
    claim_vec = vectorizer.transform([claim_clean])

    # Rank evidence passages by cosine similarity in TF-IDF space.
    scores = cosine_similarity(claim_vec, evidence_tfidf).flatten()
    top_indices = scores.argsort()[-top_k:][::-1]
    return [all_evidence_ids[i] for i in top_indices]

def build_retrieval_predictions(processed_retrieve_rows, top_k=5, verbose=True):
    # Input: list of {claim_id, claim_text}.
    # Output: the same list after in-place adding/updating `evidences`.
    # this function mutates `processed_retrieve_rows` directly.
    for row in processed_retrieve_rows:
        claim_id = row["claim_id"]
        claim_text = row["claim_text"]
        row["evidences"] = retrieve_topk_evidence_ids(claim_text, top_k=top_k)

        if verbose:
            print(f"claim_id={claim_id} claim_text={claim_text}")
            print(row["evidences"])

    return processed_retrieve_rows


# Example: retrieve top-5 evidence on dev set.
dev_retrieval_top5 = build_retrieval_predictions(processed_dev_retrieve_data, top_k=5, verbose=True)

# Quick sanity check for one sample.
print("Sample retrieval row:", dev_retrieval_top5[0])
print("Number of dev claims:", len(dev_retrieval_top5))

claim_id=claim-752 claim_text=[south australia] has the most expensive electricity in the world.
['evidence-572512', 'evidence-67732', 'evidence-147175', 'evidence-65625', 'evidence-509525']
claim_id=claim-375 claim_text=when 3 per cent of total annual global emissions of carbon dioxide are from humans and australia prod­uces 1.3 per cent of this 3 per cent, then no amount of emissions reductio­n here will have any effect on global climate.
['evidence-1140012', 'evidence-833215', 'evidence-234964', 'evidence-364039', 'evidence-1154181']
claim_id=claim-1266 claim_text=this means that the world is now 1c warmer than it was in pre-industrial times
['evidence-694262', 'evidence-911037', 'evidence-38305', 'evidence-755581', 'evidence-240270']
claim_id=claim-871 claim_text=“as it happens, zika may also be a good model of the second worrying effect — disease mutation.
['evidence-382824', 'evidence-1178347', 'evidence-1067605', 'evidence-395835', 'evidence-597838']
claim_id=claim-2164 claim_te

## 2.2 Classification

In [52]:
# load model
MODEL_NAME = "google-bert/bert-base-uncased"
LABEL_LIST = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
label2id = {name: i for i, name in enumerate(LABEL_LIST)}
id2label = {i: name for name, i in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
#
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [53]:
# Tokenize
def claims_to_dataset(processed_rows):
    return Dataset.from_dict({
        "text": [row["gold_input"] for row in processed_rows],
        "labels": [label2id[row["label"]] for row in processed_rows],
    })

train_hf = claims_to_dataset(processed_train_data)
dev_hf = claims_to_dataset(processed_dev_data)

def tokenize_batch(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

train_tok = train_hf.map(tokenize_batch, batched=True, remove_columns=["text"])
dev_tok = dev_hf.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

In [54]:


accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=preds, references=labels)


CLASSIFIER_OUTPUT_DIR = data_path.parent / "bert_classifier_ckpt"
CLASSIFIER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(CLASSIFIER_OUTPUT_DIR),

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    save_total_limit=1,
    # save_strategy="no",
    # load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=50,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
print(f"train on {model.device}")
trainer.train()

train on cuda:0


Epoch,Training Loss,Validation Loss,Accuracy
1,1.191340,1.031932,0.584416
2,0.882685,0.834425,0.675325
3,0.775406,0.929075,0.636364
4,0.662222,0.834906,0.701299
5,0.593631,0.885257,0.655844
6,0.422403,0.774897,0.688312
7,0.378283,0.833528,0.707792
8,0.277774,0.977000,0.675325
9,0.231352,0.973654,0.649351
10,0.194995,0.942962,0.655844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

TrainOutput(global_step=770, training_loss=0.5467469763446163, metrics={'train_runtime': 911.55, 'train_samples_per_second': 13.472, 'train_steps_per_second': 0.845, 'total_flos': 1873643358879552.0, 'train_loss': 0.5467469763446163, 'epoch': 10.0})

In [59]:
def build_pipeline_predictions(retrieval_records, batch_size=16):
    """Attach BERT claim_label using already-retrieved evidences."""
    # formatted_records: list of {claim_id, claim_text, label, evidences, gold_input}
    formatted_records = add_gold_input_for_records(retrieval_records)

    # Keep deterministic ordering by claim_id before batching/prediction.
    formatted_records = sorted(formatted_records, key=lambda x: x["claim_id"])
    texts = [record["gold_input"] for record in formatted_records]

    clf_pipe = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=512,
        device=device,
    )

    preds = clf_pipe(texts, batch_size=batch_size)
    labels = [pred["label"] for pred in preds]

    return {
        record["claim_id"]: {
            "claim_text": record["claim_text"],
            "claim_label": lab,
            "evidences": list(record["evidences"]),
        }
        for record, lab in zip(formatted_records, labels)
    }


def run_retrieval_then_classification(processed_retrieve_records, top_k=5, batch_size=16, verbose=False):
    """Full pipeline used for unlabeled test claims.

    Input: records with {claim_id, claim_text}.
    Output: {claim_id: {claim_label, evidences}} ready for eval format.
    """
    print(f"Retrieval start:")
    retrieval_records = build_retrieval_predictions(processed_retrieve_records, top_k=top_k, verbose=verbose)
    print(f"Retrieval End:")
    print(f"Classification:")
    return build_pipeline_predictions(retrieval_records, batch_size=batch_size)


# Teacher-forcing sanity check: use gold evidence inputs and compute dev accuracy.
_dev_pred_map = build_pipeline_predictions(processed_dev_data, batch_size=16)
print(
    "Dev accuracy (gold evidence inputs):",
    sum(
        _dev_pred_map[row["claim_id"]]["claim_label"] == dev_claims[row["claim_id"]]["claim_label"]
        for row in processed_dev_data
    )
    / len(processed_dev_data),
)

Dev accuracy (gold evidence inputs): 0.7077922077922078


# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [60]:
import numpy as np


def official_eval_like_script(predictions, groundtruth, verbose=False):
    """Same metrics as eval.py. Returns (F, A, harmonic_mean)."""
    f_scores, acc = [], []
    for claim_id, claim in sorted(groundtruth.items()):
        if claim_id not in predictions:
            continue
        pred = predictions[claim_id]
        if "claim_label" not in pred or "evidences" not in pred:
            continue

        instance_correct = 1.0 if pred["claim_label"] == claim["claim_label"] else 0.0

        evidence_correct = 0
        evidence_recall = 0.0
        evidence_precision = 0.0
        evidence_fscore = 0.0
        if isinstance(pred["evidences"], list) and len(pred["evidences"]) > 0:
            pred_set = set(pred["evidences"])
            for gr_ev in claim["evidences"]:
                if gr_ev in pred_set:
                    evidence_correct += 1
            if evidence_correct > 0:
                evidence_recall = float(evidence_correct) / len(claim["evidences"])
                evidence_precision = float(evidence_correct) / len(pred["evidences"])
                evidence_fscore = (2 * evidence_precision * evidence_recall) / (
                    evidence_precision + evidence_recall
                )

        if verbose:
            print("groundtruth =", claim)
            print("predictions =", pred)
            print("instance accuracy =", instance_correct)
            print("evidence recall =", evidence_recall)
            print("evidence precision =", evidence_precision)
            print("evidence fscore =", evidence_fscore, "\n\n")

        acc.append(instance_correct)
        f_scores.append(evidence_fscore)

    mean_f = float(np.mean(f_scores if len(f_scores) > 0 else [0.0]))
    mean_acc = float(np.mean(acc if len(acc) > 0 else [0.0]))
    if mean_f == 0.0 and mean_acc == 0.0:
        hmean = 0.0
    else:
        hmean = (2 * mean_f * mean_acc) / (mean_f + mean_acc)

    return mean_f, mean_acc, hmean


# `build_pipeline_predictions` and `run_retrieval_then_classification`
# are defined in the earlier inference cell.

# Dev: run full pipeline then evaluate with labels.
dev_pipeline_predictions = run_retrieval_then_classification(
    processed_dev_retrieve_data,
    top_k=5,
    batch_size=16,
    verbose=False,
)

F, A, H = official_eval_like_script(
    dev_pipeline_predictions, dev_claims, verbose=False
)
print("Evidence Retrieval F-score (F)    =", F)
print("Claim Classification Accuracy (A) =", A)
print("Harmonic Mean of F and A          =", H)


Evidence Retrieval F-score (F)    = 0.062208822923108656
Claim Classification Accuracy (A) = 0.33116883116883117
Harmonic Mean of F and A          = 0.10474221380668305


In [ ]:
import json

# Test (unlabeled): run retrieval + classification and save as JSON.
test_predictions = run_retrieval_then_classification(
    processed_test_data,
    top_k=5,
    batch_size=16,
    verbose=False,
)

output_path = data_path.parent / "test_predictions.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

print("Saved test predictions to:", output_path)
print("Number of test claims:", len(test_predictions))

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*